# Finger Pose Classifier

This notebook aims to use traditional classifiers to determine finger curl. Finger curl has been segmented into 5 classes:

| Class             | MCP      | PIP      | Thumb      |
|-------------------|----------|----------|------------|
| curl_curl        | curl     | curl     | ~straight  |
| curl_straight     | curl     | straight | ~straight  |
| straight_curl     | straight | curl     | ~straight  |
| straight_straight | straight | straight | ~straight  |
| thumb_curl        | straight | straight | curl       |

\
There is a separate thumb curl class because even when making a fist the the flex sensors and IMU's don't observe a substantial displacement, visually they don't seem to move too much. This thumb curl class has the thumb fully curled, with the tip of the thumb reaching into the centre of the palm.

## 1 Importing Libraries

In [23]:
# Run once to install any missing packages
import subprocess, sys
pkgs = ['scikit-learn', 'pandas', 'numpy', 'matplotlib', 'seaborn', 'scipy']
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet'] + pkgs)
print('Dependencies ready.')


Dependencies ready.


In [24]:
import os, glob, re, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import signal as scipy_signal
from scipy.stats import skew, kurtosis

from sklearn.preprocessing import LabelEncoder, StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score
)

# Classifiers
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier, StackingClassifier

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 20)
print('Imports OK.')

Imports OK.


## 2 Parameter Configuration

In [25]:
# =============================================================================
# Path to the root folder containing gesture subfolders (ThesisA.data).
# Each subfolder name becomes the class label.
# Accepts absolute paths or paths relative to this notebook.
DATA_ROOT = '../ML/NewTrainingData/Finger_MCP_PIP'   # <-- change this to point to your data folder

REPORT_OUTPUT_DIR = '/home/jestin/ThesisRepo/ML/NewReports/FingerPoseReports/FourPose'   # change if you'd like a different folder


# Labels to INCLUDE. Set to None to use ALL subfolders automatically.
# Example: INCLUDE_LABELS = ['TwoHand_L_Fist_R_Fist', 'TwoHand_L_Flat_R_Flat']
INCLUDE_LABELS = ["curl_curl", "straight_straight", "curl_straight", "straight_curl"]

# INCLUDE_LABELS = ["curl_curl", "straight_straight"]

# =============================================================================
# 1B. COLUMN SELECTION
# =============================================================================
# Choose which sensor modalities to include.
# Each flag independently includes/excludes that group of columns.

USE_YAW_PITCH_ROLL   = True    # IMU Euler angles (yaw, pitch, roll / heading)
USE_QUATERNIONS      = False   # Raw quaternion components (quat_w/x/y/z)
                               # Note: for TwoHand_L_Flat_R_Flat, quaternions are
                               #       the primary data — pitch/roll/yaw are derived.
                               #       Set True if using that label.
USE_ACCELEROMETER    = True    # Linear accelerometer (ax, ay, az)
USE_FLEX_SENSORS     = True    # Flex sensor readings (mcp_flex, pip_flex)

# Choose which body segments to include.
# Available: 'palm_prox', 'thumb', 'index', 'middle', 'ring', 'pinky', 'wrist'
# 'palm_mid'  = excluded — IMU columns exist in CSV but contain no sensor data
# 'palm_prox' = IMU on back of palm (proximal, near wrist)
# 'wrist'     = IMU at end of forearm closest to wrist
# Note: flex sensors exist for fingers only (thumb, index, middle, ring, pinky) — NOT palm
INCLUDE_SEGMENTS = [
    'palm_mid',  
    # 'palm_prox',#  excluded: IMU columns exist but contain no data
    'thumb',
    'index',
    'middle',
    'ring',
    'pinky',
    # 'wrist',
]

# Choose which hands to include
INCLUDE_HANDS = ['left', 'right']  # options: 'left', 'right', or both
FINGERS = ["thumb", "index", "middle", "ring", "pinky"]

# =============================================================================
# 1C. PREPROCESSING
# =============================================================================

# --- Resampling ---
# Resample each trial to a fixed number of time steps (handles variable-length files).
# Set to None to skip resampling (all files must then have the same row count).
RESAMPLE_TO_N_STEPS = 90       # target number of rows per trial

# --- Low-pass Butterworth filter ---
APPLY_BUTTERWORTH    = True     # Apply low-pass filter to smooth IMU noise
BUTTERWORTH_CUTOFF   = 10.0    # Cutoff frequency in Hz
BUTTERWORTH_ORDER    = 4        # Filter order
SAMPLING_RATE_HZ     = 30.0     # Approximate sampling rate of the glove

# --- Normalisation ---
# 'standard'  → zero mean, unit variance (StandardScaler) — good for SVM, LDA
# 'minmax'    → scale to [0, 1]  (MinMaxScaler) — good for KNN, neural nets
# None        → no normalisation
NORMALISATION = 'standard'

# --- Feature extraction mode ---
# 'flatten'    → use the raw resampled time series as a flat feature vector
#                (n_steps × n_channels features per sample)
# 'stats'      → extract statistical features per channel
#                (mean, std, min, max, range, RMS, skew, kurtosis)
# 'fft'        → FFT magnitude spectrum per channel
# 'stats+fft'  → combine statistical + FFT features
FEATURE_MODE = 'stats+fft'    # Recommended starting point

# =============================================================================
# 1D. TRAIN / TEST SPLIT
# =============================================================================
TEST_SIZE        = 0.2    # Fraction of data held out for testing
RANDOM_STATE     = 42
CV_FOLDS         = 5      # Number of cross-validation folds

# =============================================================================
# 1E. ALGORITHMS
# =============================================================================
# Set True/False to include/exclude each algorithm.
ALGORITHMS = {
    'SVM (RBF)'           : True,
    'SVM (Linear)'        : True,
    'Random Forest'       : True,
    'KNN'                 : True,
    'Gradient Boosting'   : False,   # Slower — enable when needed
    'Logistic Regression' : True,
    'LDA'                 : True,
    "Voting Soft (SVM+RF+KNN+LogReg)" : True,
    "Voting Soft (SVM+RF)" : True,
    "Stacking (SVM+RF+KNN+LogReg)" : True
}

print('Configuration loaded.')
print(f'  Data root  : {DATA_ROOT}')
print(f'  Hands      : {INCLUDE_HANDS}')
print(f'  Segments   : {INCLUDE_SEGMENTS}')
print(f'  Modalities : YPR={USE_YAW_PITCH_ROLL}, Quat={USE_QUATERNIONS}, Accel={USE_ACCELEROMETER}, Flex={USE_FLEX_SENSORS}')
print(f'  Features   : {FEATURE_MODE}')
print(f'  Algorithms : {[k for k,v in ALGORITHMS.items() if v]}')

Configuration loaded.
  Data root  : ../ML/NewTrainingData/Finger_MCP_PIP
  Hands      : ['left', 'right']
  Segments   : ['palm_mid', 'thumb', 'index', 'middle', 'ring', 'pinky']
  Modalities : YPR=True, Quat=False, Accel=True, Flex=True
  Features   : stats+fft
  Algorithms : ['SVM (RBF)', 'SVM (Linear)', 'Random Forest', 'KNN', 'Logistic Regression', 'LDA', 'Voting Soft (SVM+RF+KNN+LogReg)', 'Voting Soft (SVM+RF)', 'Stacking (SVM+RF+KNN+LogReg)']


## 3 Data Loading

In [26]:
# ── Build column selector from config ─────────────────────────────────────────

# All 295 columns, grouped by type for easy selection
META_COLS = [
    'run_index', 'request_id', 'request_ts',
    'right_recv_time_ms', 'right_glove_time_ms', 'right_time',
    'left_recv_time_ms',  'left_glove_time_ms',  'left_time',
    'left_hand', 'right_hand'
]

SEGMENTS_WITH_FLEX = ['thumb', 'index', 'middle', 'ring', 'pinky']  # palm has no flex sensors
IMU_LOCATIONS = ['mid', 'prox']  # distal phalanx / proximal phalanx

def build_sensor_columns(hands, segments, use_ypr, use_quat, use_accel, use_flex):
    """Return list of sensor column names matching the user config."""
    cols = []
    for hand in hands:
        for seg in segments:
            # Determine IMU sub-locations for this segment
            # palm has mid + prox; all finger segments have mid + prox; wrist has no sub-location
            if seg == 'wrist':
                prefix = f'{hand}_wrist'
                if use_ypr:
                    cols += [f'{prefix}_heading', f'{prefix}_pitch', f'{prefix}_roll']
                if use_quat:
                    cols += [f'{prefix}_quat_w', f'{prefix}_quat_x',
                             f'{prefix}_quat_y', f'{prefix}_quat_z']
                if use_accel:
                    cols += [f'{prefix}_ax', f'{prefix}_ay', f'{prefix}_az']
                # wrist has no flex sensor
            else:
                # segment has mid + prox IMU positions
                # For 'thumb','index' etc, map to e.g. 'left_thumb_mid_yaw'
                for loc in ['mid', 'prox']:
                    prefix = f'{hand}_{seg}_{loc}'
                    if use_ypr:
                        cols += [f'{prefix}_yaw', f'{prefix}_pitch', f'{prefix}_roll']
                    if use_quat:
                        cols += [f'{prefix}_quat_w', f'{prefix}_quat_x',
                                 f'{prefix}_quat_y', f'{prefix}_quat_z']
                    if use_accel:
                        cols += [f'{prefix}_ax', f'{prefix}_ay', f'{prefix}_az']
                # Flex sensors (one per finger joint combination, not per loc)
                if use_flex and seg in ['thumb', 'index', 'middle', 'ring', 'pinky']:  # palm has no flex
                    cols += [f'{hand}_{seg}_mcp_flex', f'{hand}_{seg}_pip_flex']
    return cols

# Map user-friendly segment names to CSV naming convention
seg_map = {
    'palm_mid':  'palm',   # excluded by default — no sensor data on palm_mid
    # 'palm_prox': 'palm',   # deduplicated below
    'thumb':     'thumb',
    'index':     'index',
    'middle':    'middle',
    'ring':      'ring',
    'pinky':     'pinky',
    'wrist':     'wrist',
}

# Resolve segments, de-duplicating palm if both palm_mid and palm_prox selected
resolved_segs = list(dict.fromkeys([seg_map[s] for s in INCLUDE_SEGMENTS]))

SENSOR_COLS = build_sensor_columns(
    hands    = INCLUDE_HANDS,
    segments = resolved_segs,
    use_ypr  = USE_YAW_PITCH_ROLL,
    use_quat = USE_QUATERNIONS,
    use_accel= USE_ACCELEROMETER,
    use_flex = USE_FLEX_SENSORS
)

print(f'Selected {len(SENSOR_COLS)} sensor columns.')
print('First 10:', SENSOR_COLS[:10])
print('Last 10:', SENSOR_COLS[-10:])

Selected 164 sensor columns.
First 10: ['left_palm_mid_yaw', 'left_palm_mid_pitch', 'left_palm_mid_roll', 'left_palm_mid_ax', 'left_palm_mid_ay', 'left_palm_mid_az', 'left_palm_prox_yaw', 'left_palm_prox_pitch', 'left_palm_prox_roll', 'left_palm_prox_ax']
Last 10: ['right_pinky_mid_ay', 'right_pinky_mid_az', 'right_pinky_prox_yaw', 'right_pinky_prox_pitch', 'right_pinky_prox_roll', 'right_pinky_prox_ax', 'right_pinky_prox_ay', 'right_pinky_prox_az', 'right_pinky_mcp_flex', 'right_pinky_pip_flex']


In [27]:
# ── Load all CSV files ─────────────────────────────────────────────────────────

def load_dataset(data_root, include_labels, sensor_cols):
    """Load all CSVs from label subfolders. Returns list of (df_trial, label)."""
    data_root = os.path.expanduser(data_root)
    if not os.path.isdir(data_root):
        raise FileNotFoundError(
            f"DATA_ROOT not found: '{data_root}'\n"
            "Please update DATA_ROOT in the Configuration cell to point to your "
            "ThesisA.data folder."
        )

    label_dirs = sorted([
        d for d in os.listdir(data_root)
        if os.path.isdir(os.path.join(data_root, d))
        and d not in ('PDF', 'sdb', 'idk')  # skip non-gesture folders
    ])

    if include_labels is not None:
        label_dirs = [d for d in label_dirs if d in include_labels]

    if not label_dirs:
        raise ValueError('No label folders found. Check DATA_ROOT and INCLUDE_LABELS.')

    print(f'Found {len(label_dirs)} gesture classes:')
    trials, labels = [], []

    for label in label_dirs:
        folder = os.path.join(data_root, label)
        csv_files = sorted(glob.glob(os.path.join(folder, '*.csv')))
        print(f'  [{label}]  →  {len(csv_files)} files')
        for fpath in csv_files:
            try:
                df = pd.read_csv(fpath)
                # Keep only sensor columns that exist in this file
                available = [c for c in sensor_cols if c in df.columns]
                if not available:
                    print(f'    WARNING: no matching sensor columns in {os.path.basename(fpath)}')
                    continue
                trials.append(df[available].values.astype(np.float32))
                labels.append(label)
            except Exception as e:
                print(f'    ERROR loading {os.path.basename(fpath)}: {e}')

    print(f'\nTotal trials loaded: {len(trials)}')
    return trials, labels, label_dirs

trials_raw, labels_raw, class_names = load_dataset(
    DATA_ROOT, INCLUDE_LABELS, SENSOR_COLS
)

# Class distribution
from collections import Counter
print('\nClass distribution:')
for cls, cnt in sorted(Counter(labels_raw).items()):
    print(f'  {cls}: {cnt} trials')

Found 4 gesture classes:
  [curl_curl]  →  61 files
  [curl_straight]  →  60 files
  [straight_curl]  →  60 files
  [straight_straight]  →  60 files

Total trials loaded: 241

Class distribution:
  curl_curl: 61 trials
  curl_straight: 60 trials
  straight_curl: 60 trials
  straight_straight: 60 trials


## 4. Preprocessing

In [28]:
# ── Step 1: Resample to fixed length ──────────────────────────────────────────

def resample_trial(trial, n_steps):
    """Resample a (T, C) array to (n_steps, C) using linear interpolation."""
    T, C = trial.shape
    if T == n_steps:
        return trial
    old_idx = np.linspace(0, 1, T)
    new_idx = np.linspace(0, 1, n_steps)
    resampled = np.zeros((n_steps, C), dtype=np.float32)
    for c in range(C):
        resampled[:, c] = np.interp(new_idx, old_idx, trial[:, c])
    return resampled

if RESAMPLE_TO_N_STEPS is not None:
    trials_resampled = [resample_trial(t, RESAMPLE_TO_N_STEPS) for t in trials_raw]
    print(f'Resampled all trials to {RESAMPLE_TO_N_STEPS} time steps.')
else:
    trials_resampled = trials_raw
    lengths = [t.shape[0] for t in trials_resampled]
    print(f'No resampling. Trial lengths: min={min(lengths)}, max={max(lengths)}, mean={np.mean(lengths):.1f}')
    if min(lengths) != max(lengths):
        print('  WARNING: Variable-length trials detected. Enable RESAMPLE_TO_N_STEPS '
              'or use a feature extraction mode that handles variable length.')

print(f'Trial shape: {trials_resampled[0].shape}  (time_steps × channels)')

Resampled all trials to 90 time steps.
Trial shape: (90, 164)  (time_steps × channels)


In [29]:
# ── Step 2: Butterworth low-pass filter ───────────────────────────────────────

def apply_butterworth(trials, cutoff, order, fs):
    """Apply a zero-phase Butterworth low-pass filter to each channel of each trial."""
    nyq = fs / 2.0
    norm_cutoff = cutoff / nyq
    if norm_cutoff >= 1.0:
        print(f'  WARNING: cutoff {cutoff} Hz >= Nyquist {nyq} Hz — skipping filter.')
        return trials
    b, a = scipy_signal.butter(order, norm_cutoff, btype='low', analog=False)
    filtered = []
    for t in trials:
        t_filt = scipy_signal.filtfilt(b, a, t, axis=0).astype(np.float32)
        filtered.append(t_filt)
    return filtered

if APPLY_BUTTERWORTH:
    trials_filtered = apply_butterworth(
        trials_resampled, BUTTERWORTH_CUTOFF, BUTTERWORTH_ORDER, SAMPLING_RATE_HZ
    )
    print(f'Applied Butterworth low-pass filter: {BUTTERWORTH_CUTOFF} Hz cutoff, '
          f'order {BUTTERWORTH_ORDER}, Nyquist = {SAMPLING_RATE_HZ/2} Hz')
else:
    trials_filtered = trials_resampled
    print('Butterworth filter skipped.')

Applied Butterworth low-pass filter: 10.0 Hz cutoff, order 4, Nyquist = 15.0 Hz


The data has been resampled to be of size 90 rows x 176 sensor columns, and a butterworth filter has been applied to all the sensor columns to smooth the readings.

Now we need to split up each of the trials by finger. We need a dataset for each finger on each hand.

In [30]:
# ── Split processed trials into 10 finger datasets ─────────────────────────────

FINGERS = ['thumb', 'index', 'middle', 'ring', 'pinky']
HANDS = ['left', 'right']

def get_finger_columns(sensor_cols, hand, finger, palm_segment='palm_mid'):
    """
    For one hand/finger pair, return:
      - that finger's columns
      - that hand's palm columns
    using the already-selected SENSOR_COLS ordering.
    """
    finger_prefix = f"{hand}_{finger}_"
    palm_prefix = f"{hand}_{palm_segment}_"

    finger_cols = [c for c in sensor_cols if c.startswith(finger_prefix)]
    palm_cols = [c for c in sensor_cols if c.startswith(palm_prefix)]

    return palm_cols + finger_cols

def split_trials_by_finger(trials, labels, sensor_cols, hands=None, fingers=None, palm_segment='palm_mid'):
    """
    Split list of processed trial arrays into 10 datasets:
      (left/right) x (thumb/index/middle/ring/pinky)

    Returns
    -------
    finger_data : dict
        key   = (hand, finger)
        value = {
            'raw': list of np.ndarray with shape (T, C_finger),
            'labels': list of class labels aligned with raw,
            'columns': ordered column names used for that dataset
        }
    """
    hands = HANDS if hands is None else hands
    fingers = FINGERS if fingers is None else fingers

    col_index = {c: i for i, c in enumerate(sensor_cols)}
    finger_data = {}

    for hand in hands:
        for finger in fingers:
            key = (hand, finger)
            cols = get_finger_columns(sensor_cols, hand, finger, palm_segment=palm_segment)
            idxs = [col_index[c] for c in cols if c in col_index]

            finger_data[key] = {
                "raw": [trial[:, idxs] for trial in trials],
                "labels": list(labels),
                "columns": cols,
            }

    return finger_data

finger_trials = split_trials_by_finger(
    trials_filtered,
    labels_raw,
    SENSOR_COLS,
    hands=INCLUDE_HANDS,
    fingers=FINGERS,
    palm_segment='palm_mid'   # change to 'palm_prox' if that is what you really want
)

print("Built per-finger datasets:")
for key in finger_trials:
    first_shape = finger_trials[key]['raw'][0].shape if len(finger_trials[key]['raw']) else None
    print(f"{key}: {len(finger_trials[key]['raw'])} trials, shape per trial = {first_shape}, channels = {len(finger_trials[key]['columns'])}")

# Inspect columns for right thumb and right ring
print("\nRight Thumb columns:")
print(finger_trials[('right', 'thumb')]['columns'])
print("\nRight Ring columns:")
print(finger_trials[('right', 'ring')]['columns'])

Built per-finger datasets:
('left', 'thumb'): 241 trials, shape per trial = (90, 20), channels = 20
('left', 'index'): 241 trials, shape per trial = (90, 20), channels = 20
('left', 'middle'): 241 trials, shape per trial = (90, 20), channels = 20
('left', 'ring'): 241 trials, shape per trial = (90, 20), channels = 20
('left', 'pinky'): 241 trials, shape per trial = (90, 20), channels = 20
('right', 'thumb'): 241 trials, shape per trial = (90, 20), channels = 20
('right', 'index'): 241 trials, shape per trial = (90, 20), channels = 20
('right', 'middle'): 241 trials, shape per trial = (90, 20), channels = 20
('right', 'ring'): 241 trials, shape per trial = (90, 20), channels = 20
('right', 'pinky'): 241 trials, shape per trial = (90, 20), channels = 20

Right Thumb columns:
['right_palm_mid_yaw', 'right_palm_mid_pitch', 'right_palm_mid_roll', 'right_palm_mid_ax', 'right_palm_mid_ay', 'right_palm_mid_az', 'right_thumb_mid_yaw', 'right_thumb_mid_pitch', 'right_thumb_mid_roll', 'right_thum

In [31]:
# ── Data augmentation utilities for time-series trials ─────────────────────────

def add_gaussian_noise(trial, noise_std=0.01):
    """
    trial: (T, C) np.ndarray
    Adds zero-mean Gaussian noise with given std (relative to channel scale).
    """
    # scale noise per channel based on its std, to keep it proportional
    channel_std = np.std(trial, axis=0, keepdims=True) + 1e-8
    noise = np.random.normal(loc=0.0, scale=noise_std, size=trial.shape) * channel_std
    return trial + noise.astype(np.float32)

def random_time_warp(trial, max_scale=0.1):
    """
    Randomly stretch/compress time axis by up to ±max_scale and resample to original length.
    This preserves (T, C) shape.
    """
    T, C = trial.shape
    # sample scale factor in [1-max_scale, 1+max_scale]
    scale = 1.0 + np.random.uniform(-max_scale, max_scale)
    new_T = int(round(T * scale))
    old_idx = np.linspace(0, 1, T)
    new_idx = np.linspace(0, 1, new_T)

    warped = np.zeros((new_T, C), dtype=np.float32)
    for c in range(C):
        warped[:, c] = np.interp(new_idx, old_idx, trial[:, c])

    # resample back to length T
    return resample_trial(warped, T)  # reuses your existing resample_trial()[file:191]

def random_channel_scaling(trial, scale_range=(0.9, 1.1)):
    """
    Randomly scale each channel by a factor in scale_range.
    """
    T, C = trial.shape
    scales = np.random.uniform(scale_range[0], scale_range[1], size=(1, C))
    return (trial * scales).astype(np.float32)

In [32]:
# ── Build augmented copies of each finger dataset ──────────────────────────────

def augment_finger_data(
        finger_data,
        n_aug_per_trial=1,
        use_noise=True,
        use_time_warp=True,
        use_scaling=False):
    """
    For each (hand, finger), create augmented trials and labels.

    n_aug_per_trial: how many augmented versions to create per original trial.
    Returns a NEW dict with original + augmented data mixed together.
    """
    aug_finger_data = {}

    for key, fd in finger_data.items():
        raw_trials = fd["raw"]
        labels = fd["labels"]
        cols = fd["columns"]

        new_raw = list(raw_trials)      # start with originals
        new_labels = list(labels)

        for trial, label in zip(raw_trials, labels):
            for _ in range(n_aug_per_trial):
                t_aug = trial.copy()

                if use_time_warp:
                    t_aug = random_time_warp(t_aug, max_scale=0.1)

                if use_noise:
                    t_aug = add_gaussian_noise(t_aug, noise_std=0.02)

                if use_scaling:
                    t_aug = random_channel_scaling(t_aug, scale_range=(0.9, 1.1))

                new_raw.append(t_aug)
                new_labels.append(label)

        aug_finger_data[key] = {
            "raw": new_raw,
            "labels": new_labels,
            "columns": cols,
        }

    return aug_finger_data


def augment_raw_trials(
        raw_trials,
        labels,
        n_aug_per_trial=1,
        use_noise=True,
        use_time_warp=True,
        use_scaling=False):
    """
    Augment a set of raw trials and labels.
    Returns augmented trials and corresponding labels.
    """
    new_raw = list(raw_trials)
    new_labels = list(labels)

    for trial, label in zip(raw_trials, labels):
        for _ in range(n_aug_per_trial):
            t_aug = trial.copy()

            if use_time_warp:
                t_aug = random_time_warp(t_aug, max_scale=0.1)

            if use_noise:
                t_aug = add_gaussian_noise(t_aug, noise_std=0.02)

            if use_scaling:
                t_aug = random_channel_scaling(t_aug, scale_range=(0.9, 1.1))

            new_raw.append(t_aug)
            new_labels.append(label)

    return new_raw, new_labels

# `finger_data_aug` is created later after raw train/test splitting to avoid leakage.


mid         = 3x accel + 3x ypr             = 6\
prox        = 3x accel + 3x ypr             = 6\
palm_mid    = 3x accel + 3x ypr             = 6\
flex        = mcp + pip                     = 2\
\
total       = mid + prox + palm_mid + flex  = 20\

## 5. Feature Extraction

In [33]:
# ── Step 3: Feature extraction ────────────────────────────────────────────────

def extract_stats(trial):
    """Statistical features per channel: mean, std, min, max, range, RMS, skew, kurtosis."""
    feats = np.concatenate([
        trial.mean(axis=0),
        trial.std(axis=0),
        trial.min(axis=0),
        trial.max(axis=0),
        trial.max(axis=0) - trial.min(axis=0),          # range
        np.sqrt((trial**2).mean(axis=0)),                # RMS
        skew(trial, axis=0).astype(np.float32),
        kurtosis(trial, axis=0).astype(np.float32),
    ])
    return feats


def extract_fft(trial, fs):
    """FFT magnitude features per channel (first half of spectrum)."""
    n = trial.shape[0]
    fft_mag = np.abs(np.fft.rfft(trial, axis=0))  # shape: (n//2+1, C)
    return fft_mag.flatten().astype(np.float32)


def extract_features(trials, mode, fs):
    X = []
    for t in trials:
        if mode == 'flatten':
            feats = t.flatten()
        elif mode == 'stats':
            feats = extract_stats(t)
        elif mode == 'fft':
            feats = extract_fft(t, fs)
        elif mode == 'stats+fft':
            feats = np.concatenate([extract_stats(t), extract_fft(t, fs)])
        else:
            raise ValueError(f'Unknown FEATURE_MODE: {mode}')
        X.append(feats)
    return np.array(X, dtype=np.float32)


# print(type(finger_trials))

finger_data_aug = {}

for key in finger_trials:
    raw_trials = finger_trials[key]['raw']
    labels = finger_trials[key]['labels']
    cols = finger_trials[key]['columns']

    raw_train, raw_test, labels_train, labels_test = train_test_split(
        raw_trials,
        labels,
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE,
        stratify=labels,
    )

    raw_train_aug, labels_train_aug = augment_raw_trials(
        raw_train,
        labels_train,
        n_aug_per_trial=3,
        use_noise=True,
        use_time_warp=True,
        use_scaling=False,
    )

    print(f"=== Dataset: {key} ===")
    print(f"  Original samples: {len(raw_trials)}")
    print(f"  Train originals: {len(raw_train)}")
    print(f"  Train augmented: {len(raw_train_aug)}")
    print(f"  Test samples: {len(raw_test)}")

    le = LabelEncoder()
    le.fit(labels)

    X_train = extract_features(raw_train_aug, FEATURE_MODE, SAMPLING_RATE_HZ)
    X_test = extract_features(raw_test, FEATURE_MODE, SAMPLING_RATE_HZ)

    for dataset_name, X in [('train', X_train), ('test', X_test)]:
        bad = np.isnan(X) | np.isinf(X)
        if bad.any():
            print(f'WARNING: {bad.sum()} NaN/Inf values found in {dataset_name} set — replacing with 0.')
        if dataset_name == 'train':
            X_train = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)
        else:
            X_test = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)

    print(f'Feature matrix shapes: train={X_train.shape}, test={X_test.shape}')

    y_train = le.transform(labels_train_aug)
    y_test = le.transform(labels_test)

    finger_data_aug[key] = {
        'raw_train': raw_train_aug,
        'raw_test': raw_test,
        'labels_train': labels_train_aug,
        'labels_test': labels_test,
        'columns': cols,
        'X_train': X_train,
        'X_test': X_test,
        'y_train': y_train,
        'y_test': y_test,
        'le': le,
    }


=== Dataset: ('left', 'thumb') ===
  Original samples: 241
  Train originals: 192
  Train augmented: 768
  Test samples: 49
Feature matrix shapes: train=(768, 1080), test=(49, 1080)
=== Dataset: ('left', 'index') ===
  Original samples: 241
  Train originals: 192
  Train augmented: 768
  Test samples: 49
Feature matrix shapes: train=(768, 1080), test=(49, 1080)
=== Dataset: ('left', 'middle') ===
  Original samples: 241
  Train originals: 192
  Train augmented: 768
  Test samples: 49
Feature matrix shapes: train=(768, 1080), test=(49, 1080)
=== Dataset: ('left', 'ring') ===
  Original samples: 241
  Train originals: 192
  Train augmented: 768
  Test samples: 49
Feature matrix shapes: train=(768, 1080), test=(49, 1080)
=== Dataset: ('left', 'pinky') ===
  Original samples: 241
  Train originals: 192
  Train augmented: 768
  Test samples: 49
Feature matrix shapes: train=(768, 1080), test=(49, 1080)
=== Dataset: ('right', 'thumb') ===
  Original samples: 241
  Train originals: 192
  Train

We've split the data by finger, we've extracted statistical and fft features from each channel. As result, we've flattened the data into a 1D dataset of 160 features.

## 6. Train | Test Split

In [34]:
for key in finger_data_aug:
    print(f"\n=== Dataset: {key} ===")
    # print(f"{key} X shape:", finger_data_aug[key]['X'].shape)
    # print(f"{key} y shape:", finger_data_aug[key]['y'].shape)
    # print(f"{key} Classes:", list(finger_data_aug[key]['le'].classes_))

    # X_train, X_test, y_train, y_test = train_test_split(
    # finger_data_aug[key]['X'], finger_data_aug[key]['y'], test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=finger_data_aug[key]['y']
    # )

    if NORMALISATION == 'standard':
        scaler = StandardScaler()
    elif NORMALISATION == 'minmax':
        scaler = MinMaxScaler()
    else:
        scaler = None

    if scaler is not None:
        X_train = scaler.fit_transform(finger_data_aug[key]['X_train'])
        X_test  = scaler.transform(finger_data_aug[key]['X_test'])
        print(f'Applied {NORMALISATION} normalisation.')
    else:
        print('No normalisation applied.')

    finger_data_aug[key].update({'X_train': X_train, 'X_test': X_test, 'y_train': y_train, 'y_test': y_test})

    print(f'Train: {X_train.shape[0]} samples | Test: {X_test.shape[0]} samples')


=== Dataset: ('left', 'thumb') ===
Applied standard normalisation.
Train: 768 samples | Test: 49 samples

=== Dataset: ('left', 'index') ===
Applied standard normalisation.
Train: 768 samples | Test: 49 samples

=== Dataset: ('left', 'middle') ===
Applied standard normalisation.
Train: 768 samples | Test: 49 samples

=== Dataset: ('left', 'ring') ===
Applied standard normalisation.
Train: 768 samples | Test: 49 samples

=== Dataset: ('left', 'pinky') ===
Applied standard normalisation.
Train: 768 samples | Test: 49 samples

=== Dataset: ('right', 'thumb') ===
Applied standard normalisation.
Train: 768 samples | Test: 49 samples

=== Dataset: ('right', 'index') ===
Applied standard normalisation.
Train: 768 samples | Test: 49 samples

=== Dataset: ('right', 'middle') ===
Applied standard normalisation.
Train: 768 samples | Test: 49 samples

=== Dataset: ('right', 'ring') ===
Applied standard normalisation.
Train: 768 samples | Test: 49 samples

=== Dataset: ('right', 'pinky') ===
Appli

In [35]:
# ── Define classifiers from config ────────────────────────────────────────────

CLASSIFIER_DEFS = {
    'SVM (RBF)':           SVC(kernel='rbf', C=10, gamma='scale', random_state=RANDOM_STATE, probability=True),
    'SVM (Linear)':        SVC(kernel='linear', C=1, random_state=RANDOM_STATE, probability=True),
    'Random Forest':       RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1),
    'KNN':                 KNeighborsClassifier(n_neighbors=5, metric='euclidean'),
    'Gradient Boosting':   GradientBoostingClassifier(n_estimators=100, random_state=RANDOM_STATE),
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    'LDA':                 LinearDiscriminantAnalysis(),
}

# Base estimators to use in ensembles (reusing the above definitions)
base_estimators = [
    ("svm_rbf", CLASSIFIER_DEFS["SVM (RBF)"]),
    ("rf", CLASSIFIER_DEFS["Random Forest"]),
    ("knn", CLASSIFIER_DEFS["KNN"]),
    ("logreg", CLASSIFIER_DEFS["Logistic Regression"]),
]

# Voting ensemble (soft voting)
CLASSIFIER_DEFS["Voting Soft (SVM+RF+KNN+LogReg)"] = VotingClassifier(
    estimators=base_estimators,
    voting="soft",     # use predict_proba and average
    n_jobs=-1,
)

CLASSIFIER_DEFS["Voting Soft (SVM+RF)"] = VotingClassifier(
    estimators=base_estimators,
    voting="soft",     # use predict_proba and average
    n_jobs=-1,
)

# Stacking ensemble
CLASSIFIER_DEFS["Stacking (SVM+RF+KNN+LogReg)"] = StackingClassifier(
    estimators=base_estimators,
    final_estimator=LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    cv=CV_FOLDS,       # e.g., 5, consistent with your config
    passthrough=False,
    n_jobs=-1,
)

active_classifiers = {k: v for k, v in CLASSIFIER_DEFS.items() if ALGORITHMS.get(k, False)}
print(f'Running {len(active_classifiers)} algorithms: {list(active_classifiers.keys())}')

Running 9 algorithms: ['SVM (RBF)', 'SVM (Linear)', 'Random Forest', 'KNN', 'Logistic Regression', 'LDA', 'Voting Soft (SVM+RF+KNN+LogReg)', 'Voting Soft (SVM+RF)', 'Stacking (SVM+RF+KNN+LogReg)']


In [36]:
# ── Print model parameters for all active classifiers ─────────────────────────
# Shows every hyperparameter and (after training) learned model properties.

PARAM_DESCRIPTIONS = {
    # SVM
    'C':               'Regularisation — higher = fits training data more tightly, lower = wider margin (less overfit)',
    'kernel':          'Decision boundary shape: rbf=curved (non-linear), linear=flat hyperplane',
    'gamma':           'RBF kernel width — scale=1/(n_features*X.var()), auto=1/n_features. Higher=more complex boundary',
    # Random Forest
    'n_estimators':    'Number of decision trees in the forest — more trees = more stable, slower',
    'max_depth':       'Max tree depth — None=grow until pure leaves (may overfit)',
    'min_samples_split':'Min samples required to split a node — higher=simpler trees',
    'min_samples_leaf': 'Min samples required at a leaf — higher=smoother decision boundary',
    'max_features':    'Features considered per split — sqrt=sqrt(n_features), None=all features',
    # KNN
    'n_neighbors':     'Number of nearest neighbours to vote — lower=more sensitive to noise, higher=smoother boundary',
    'metric':          'Distance measure used to find neighbours',
    'weights':         'uniform=all neighbours equal vote, distance=closer neighbours weighted more',
    # Logistic Regression
    'max_iter':        'Maximum solver iterations — increase if convergence warning appears',
    'C':               'Inverse regularisation strength — higher=less regularisation (fits training harder)',
    'solver':          'Optimisation algorithm used to fit the model',
    'multi_class':     'Strategy for multi-class: auto selects ovr or multinomial',
    # LDA
    'solver':          'svd=no matrix inversion (stable), lsqr/eigen=faster but less stable',
    'shrinkage':       'Regularisation on covariance matrix — None=no shrinkage',
    'n_components':    'Dimensions to reduce to — None=min(n_classes-1, n_features)',
    # Gradient Boosting
    'learning_rate':   'Shrinks each tree contribution — lower=more trees needed but better generalisation',
    'subsample':       'Fraction of samples per tree — <1.0 adds randomness (stochastic boosting)',
}

print('=' * 70)
print('CLASSIFIER PARAMETERS')
print('=' * 70)

for name, clf in active_classifiers.items():
    params = clf.get_params()
    print(f'\n┌─ {name} ─────────────────────────────────────────')
    print(f'│  Type: {type(clf).__name__}')
    print(f'│')
    for param, value in params.items():
        desc = PARAM_DESCRIPTIONS.get(param, '')
        desc_str = f'  # {desc}' if desc else ''
        print(f'│  {param:<22} = {str(value):<15}{desc_str}')
    print(f'└──────────────────────────────────────────────────')

# ── Post-training learned properties (run after Section 4 training cell) ─────
print()
print('=' * 70)
print('POST-TRAINING LEARNED PROPERTIES  (run after Section 4)')
print('=' * 70)

if 'results' not in dir():
    print('  Classifiers not yet trained — run Section 4 first.')
else:
    for name, res in results.items():
        clf = res['clf']
        print(f'\n  {name}:')

        if hasattr(clf, 'support_vectors_'):
            n_sv = clf.support_vectors_.shape[0]
            pct  = 100 * n_sv / len(X_train)
            print(f'    Support vectors : {n_sv} / {len(X_train)} training samples ({pct:.1f}%)')
            print(f'    (Low % = confident, wide margin. High % = decision boundary uncertain)')

        if hasattr(clf, 'n_iter_'):
            print(f'    Iterations      : {clf.n_iter_}')

        if hasattr(clf, 'feature_importances_'):
            fi   = clf.feature_importances_
            top3 = fi.argsort()[::-1][:3]
            print(f'    Feature importances (top 3 feature indices): {top3}')
            print(f'    Top 3 importance values: {fi[top3].round(4)}')

        if hasattr(clf, 'coef_'):
            coef = clf.coef_
            print(f'    Coefficient matrix shape: {coef.shape}  ({coef.shape[0]} classes × {coef.shape[1]} features)')
            print(f'    Coef magnitude  : mean={abs(coef).mean():.4f}, max={abs(coef).max():.4f}')

        if hasattr(clf, 'scalings_'):
            print(f'    LDA scalings shape: {clf.scalings_.shape}  ({clf.scalings_.shape[1]} discriminant axes)')
            print(f'    Explained variance ratio: {clf.explained_variance_ratio_.round(4) if hasattr(clf, "explained_variance_ratio_") else "N/A"}')
        if hasattr(clf, 'estimators_') and name == 'Random Forest':
            depths = [e.get_depth() for e in clf.estimators_]
            print(f'    Trees grown     : {len(clf.estimators_)}')
            print(f'    Tree depth      : mean={sum(depths)/len(depths):.1f}, max={max(depths)}, min={min(depths)}')


CLASSIFIER PARAMETERS

┌─ SVM (RBF) ─────────────────────────────────────────
│  Type: SVC
│
│  C                      = 10               # Inverse regularisation strength — higher=less regularisation (fits training harder)
│  break_ties             = False          
│  cache_size             = 200            
│  class_weight           = None           
│  coef0                  = 0.0            
│  decision_function_shape = ovr            
│  degree                 = 3              
│  gamma                  = scale            # RBF kernel width — scale=1/(n_features*X.var()), auto=1/n_features. Higher=more complex boundary
│  kernel                 = rbf              # Decision boundary shape: rbf=curved (non-linear), linear=flat hyperplane
│  max_iter               = -1               # Maximum solver iterations — increase if convergence warning appears
│  probability            = True           
│  random_state           = 42             
│  shrinking              = True           


In [37]:
finger_classifiers = {}
accuracies = {}
per_class_tables = {}       # per (finger, classifier): long form
per_class_tables_wide = {}  # per (finger): rows=class, cols=classifiers

for key in finger_data_aug:
    X_train = finger_data_aug[key]['X_train']
    y_train = finger_data_aug[key]['y_train']
    X_test  = finger_data_aug[key]['X_test']
    y_test  = finger_data_aug[key]['y_test']
    le      = finger_data_aug[key]['le']   # LabelEncoder for this finger

    results = {}
    accuracies[key] = {}
    per_class_tables[key] = {}

    class_ids = np.arange(len(le.classes_))
    class_names = le.classes_

    # temp dict to collect per-class accuracy arrays per classifier
    per_class_acc = {}

    for name, clf in active_classifiers.items():
        clf.fit(X_train, y_train)
        y_pred = clf.predict(X_test)

        # Overall accuracy
        acc = accuracy_score(y_test, y_pred)
        accuracies[key][name] = acc

        # Confusion matrix using all classes in fixed order
        cm = confusion_matrix(y_test, y_pred, labels=class_ids)

        correct = np.diag(cm)
        total = cm.sum(axis=1)

        # avoid divide-by-zero just in case
        class_acc = np.divide(
            correct, total,
            out=np.zeros_like(correct, dtype=float),
            where=total != 0
        )

        # store a nice dataframe for this finger + classifier (long form)
        df_class = pd.DataFrame({
            'class': class_names,
            'correct': correct,
            'total': total,
            'per_class_accuracy': class_acc
        })

        per_class_tables[key][name] = df_class
        per_class_acc[name] = class_acc

        results[name] = {
            'clf': clf,
            'accuracy': acc,
            'per_class_accuracy': df_class
        }

    finger_classifiers[key] = results

    # ---- Build wide table for this finger: rows = class, cols = classifiers ----
    df_wide = pd.DataFrame({'Finger': class_names})
    for clf_name, class_acc in per_class_acc.items():
        df_wide[clf_name] = class_acc
    per_class_tables_wide[key] = df_wide

# overall accuracy table
df_overall = pd.DataFrame(accuracies).T
print("\nOverall Accuracies Table:")
print(df_overall.to_string())

# print wide per-class tables in the Finger x classifier format you want
for key, df_wide in per_class_tables_wide.items():
    print("\n" + "="*80)
    print(f"Per-class accuracy table (rows = class, cols = classifier) for dataset: {key}")
    print("="*80)
    print(df_wide.to_string(index=False))


Overall Accuracies Table:
              SVM (RBF)  SVM (Linear)  Random Forest       KNN  Logistic Regression       LDA  Voting Soft (SVM+RF+KNN+LogReg)  Voting Soft (SVM+RF)  Stacking (SVM+RF+KNN+LogReg)
left  thumb    0.734694      0.734694       0.836735  0.571429             0.755102  0.489796                         0.755102              0.755102                      0.714286
      index    0.775510      0.734694       0.857143  0.714286             0.714286  0.612245                         0.775510              0.775510                      0.755102
      middle   0.775510      0.795918       0.897959  0.693878             0.714286  0.755102                         0.775510              0.775510                      0.734694
      ring     0.714286      0.775510       0.918367  0.591837             0.775510  0.734694                         0.795918              0.795918                      0.775510
      pinky    0.673469      0.714286       0.877551  0.571429             0.7

In [38]:
finger_classifiers = {}
accuracies = {}

for key in finger_data_aug:
    # print(f"\n=== Training classifiers for dataset: {key} ===")
    X_train = finger_data_aug[key]['X_train']
    y_train = finger_data_aug[key]['y_train']
    X_test  = finger_data_aug[key]['X_test']
    y_test  = finger_data_aug[key]['y_test']

    results = {}
    for name, clf in active_classifiers.items():
        # print(f'\nTraining {name}...')
        clf.fit(X_train, y_train)
        y_pred = clf.predict(X_test)
        acc = accuracy_score(y_test, y_pred)
        # print(f'  Test Accuracy: {acc:.4f}')
        results[name] = {'clf': clf, 'accuracy': acc}
        accuracies.setdefault(key, {})[name] = acc

    finger_classifiers[key] = results

# Display accuracies in tabular form

df = pd.DataFrame(accuracies).T
print("\nAccuracies Table:")
print(df.to_string())


Accuracies Table:
              SVM (RBF)  SVM (Linear)  Random Forest       KNN  Logistic Regression       LDA  Voting Soft (SVM+RF+KNN+LogReg)  Voting Soft (SVM+RF)  Stacking (SVM+RF+KNN+LogReg)
left  thumb    0.734694      0.734694       0.836735  0.571429             0.755102  0.489796                         0.755102              0.755102                      0.714286
      index    0.775510      0.734694       0.857143  0.714286             0.714286  0.612245                         0.775510              0.775510                      0.755102
      middle   0.775510      0.795918       0.897959  0.693878             0.714286  0.755102                         0.775510              0.775510                      0.734694
      ring     0.714286      0.775510       0.918367  0.591837             0.775510  0.734694                         0.795918              0.795918                      0.775510
      pinky    0.673469      0.714286       0.877551  0.571429             0.755102  0

## 7. Test on a New CSV File

This section lets you feed an arbitrary single-trial CSV (e.g. `glove_data_L_Static_Double_Closed_Fist_3s_*.csv`,
`glove_data_L_Static_Double_Open_Palm_3s_*.csv`, or any file with the same 295-column schema as the
`glove_data_L_Finger_MCP_PIP_*` files) and predict the curl class of every finger
on every selected hand using the classifiers trained above.

Just point `TEST_CSV_PATH` at the file you want to classify and run the cells below.


In [39]:
# ── 7.1 Refit per-finger scalers so they can be reused at test time ─────────────
# Cell 23 fits a StandardScaler/MinMaxScaler per finger but only keeps the LAST
# one in scope (loop-local variable). To predict on new data we need each
# finger's scaler, so we refit one per (hand, finger) using the SAME training
# features cell 23 fit on, and stash them in `finger_scalers`.

finger_scalers = {}

for key in finger_data_aug:
    # Re-extract features from the (already augmented) raw training trials so we
    # have the unscaled feature matrix to refit the scaler on. This reproduces
    # exactly what cell 20 → cell 23 did.
    raw_train_aug = finger_data_aug[key]['raw_train']
    X_train_unscaled = extract_features(raw_train_aug, FEATURE_MODE, SAMPLING_RATE_HZ)
    X_train_unscaled = np.nan_to_num(X_train_unscaled, nan=0.0, posinf=0.0, neginf=0.0)

    if NORMALISATION == 'standard':
        scaler = StandardScaler()
    elif NORMALISATION == 'minmax':
        scaler = MinMaxScaler()
    else:
        scaler = None

    if scaler is not None:
        scaler.fit(X_train_unscaled)

    finger_scalers[key] = scaler

print(f'Stored scalers for {len(finger_scalers)} (hand, finger) datasets.')


Stored scalers for 10 (hand, finger) datasets.


In [40]:
# ── 7.2 Inference helper: CSV path → per-finger curl predictions ────────────────

def predict_finger_curls(
        csv_path,
        classifier_name=None,
        hands=None,
        fingers=None,
        sensor_cols=None,
        palm_segment='palm_mid',
        return_proba=True,
):
    """
    Run the full preprocessing → feature → classify pipeline on a single CSV
    and return the predicted curl class for every (hand, finger).

    Parameters
    ----------
    csv_path : str
        Path to a CSV file with the same column schema as the training CSVs
        (e.g. glove_data_L_Static_Double_Closed_Fist_*.csv).
    classifier_name : str, optional
        Which trained classifier to use (e.g. 'SVM (RBF)', 'Random Forest', …).
        Defaults to the best-accuracy classifier per finger.
    hands : list[str], optional
        Subset of ['left', 'right']. Defaults to INCLUDE_HANDS.
    fingers : list[str], optional
        Subset of ['thumb','index','middle','ring','pinky']. Defaults to FINGERS.
    sensor_cols : list[str], optional
        Sensor column ordering. Defaults to the global SENSOR_COLS used during training.
    palm_segment : str
        Same palm segment used during training (default 'palm_mid').
    return_proba : bool
        If True, also return the predicted-class probability when available.

    Returns
    -------
    pandas.DataFrame with columns:
        hand, finger, classifier, predicted_class [, confidence]
    """
    hands = INCLUDE_HANDS if hands is None else hands
    fingers = FINGERS if fingers is None else fingers
    sensor_cols = SENSOR_COLS if sensor_cols is None else sensor_cols

    csv_path = os.path.expanduser(csv_path)
    if not os.path.isfile(csv_path):
        raise FileNotFoundError(f'Test CSV not found: {csv_path}')

    df = pd.read_csv(csv_path)

    # 1. Restrict to the same sensor columns used at training time
    available = [c for c in sensor_cols if c in df.columns]
    missing = [c for c in sensor_cols if c not in df.columns]
    if missing:
        print(f'  WARNING: {len(missing)} expected sensor columns are missing in this CSV. '
              f'First few: {missing[:5]}')
    if not available:
        raise ValueError('No matching sensor columns found in the test CSV.')

    trial = df[available].values.astype(np.float32)
    print(f'Loaded {os.path.basename(csv_path)} → raw shape {trial.shape}')

    # 2. Resample to the same fixed length used at training time
    if RESAMPLE_TO_N_STEPS is not None:
        trial = resample_trial(trial, RESAMPLE_TO_N_STEPS)

    # 3. Apply Butterworth filter (same params as training)
    if APPLY_BUTTERWORTH:
        trial = apply_butterworth(
            [trial], BUTTERWORTH_CUTOFF, BUTTERWORTH_ORDER, SAMPLING_RATE_HZ
        )[0]

    # 4. Split into per-finger trials using the SAME column ordering as training
    col_index = {c: i for i, c in enumerate(available)}
    rows = []

    for hand in hands:
        for finger in fingers:
            key = (hand, finger)
            if key not in finger_classifiers:
                # this finger wasn't trained (e.g. user excluded a hand)
                continue

            # Reproduce get_finger_columns() but restricted to columns
            # that actually exist in this CSV (palm + finger).
            train_cols = finger_trials[key]['columns']
            cols_for_finger = [c for c in train_cols if c in col_index]
            if len(cols_for_finger) != len(train_cols):
                print(f'  WARNING: {key} expected {len(train_cols)} cols, '
                      f'found {len(cols_for_finger)} in test CSV.')
            if not cols_for_finger:
                continue
            idxs = [col_index[c] for c in cols_for_finger]
            finger_trial = trial[:, idxs]

            # 5. Feature extraction (same FEATURE_MODE as training)
            X = extract_features([finger_trial], FEATURE_MODE, SAMPLING_RATE_HZ)
            X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)

            # 6. Apply the per-finger scaler that was fit on training data
            scaler = finger_scalers.get(key)
            if scaler is not None:
                X = scaler.transform(X)

            # 7. Pick which trained classifier to use
            results = finger_classifiers[key]
            if classifier_name is None:
                # default → highest-accuracy classifier on the held-out test set
                clf_name = max(results, key=lambda n: results[n]['accuracy'])
            else:
                if classifier_name not in results:
                    raise KeyError(
                        f'Classifier {classifier_name!r} not found for {key}. '
                        f'Available: {list(results)}'
                    )
                clf_name = classifier_name
            clf = results[clf_name]['clf']
            le = finger_data_aug[key]['le']

            y_pred = clf.predict(X)
            pred_label = le.inverse_transform(y_pred)[0]

            row = {
                'hand': hand,
                'finger': finger,
                'classifier': clf_name,
                'predicted_class': pred_label,
            }

            if return_proba and hasattr(clf, 'predict_proba'):
                proba = clf.predict_proba(X)[0]
                row['confidence'] = float(proba.max())
                # also expose every class probability for inspection
                for cls_name, p in zip(le.classes_, proba):
                    row[f'p({cls_name})'] = float(p)

            rows.append(row)

    return pd.DataFrame(rows)


In [41]:
# ── 7.3 Run on a test CSV ───────────────────────────────────────────────────────
# Point this at the file you want to classify. Any CSV with the same 295-column
# schema as the training data works — e.g. the Static_Double_Closed_Fist /
# Static_Double_Open_Palm files, or any other Finger_MCP_PIP_*.csv.

TEST_CSV_PATH = '/home/jestin/ThesisRepo/ML/NewTrainingData/Static/Double_Closed_Fist/glove_data_L_Static_Double_Closed_Fist_3s_50_2026-05-02_00-23-41.csv'

In [42]:


# Set to a specific name (e.g. 'SVM (RBF)') to force one classifier across all
# fingers, or leave as None to pick the best-accuracy classifier per finger.
TEST_CLASSIFIER = None

predictions_df = predict_finger_curls(
    TEST_CSV_PATH,
    classifier_name=TEST_CLASSIFIER,
    return_proba=True,
)

# Compact summary view
summary_cols = ['hand', 'finger', 'classifier', 'predicted_class']
if 'confidence' in predictions_df.columns:
    summary_cols.append('confidence')

print(f"\nPredictions for: {os.path.basename(TEST_CSV_PATH)}")
print("=" * 80)
print(predictions_df[summary_cols].to_string(index=False))

# Full per-class probability table (one row per finger)
proba_cols = [c for c in predictions_df.columns if c.startswith('p(')]
if proba_cols:
    print("\nPer-class probabilities:")
    print("=" * 80)
    print(predictions_df[['hand', 'finger'] + proba_cols].to_string(index=False))


Loaded glove_data_L_Static_Double_Closed_Fist_3s_50_2026-05-02_00-23-41.csv → raw shape (92, 164)

Predictions for: glove_data_L_Static_Double_Closed_Fist_3s_50_2026-05-02_00-23-41.csv
 hand finger                   classifier predicted_class  confidence
 left  thumb                Random Forest       curl_curl    0.430000
 left  index                Random Forest   curl_straight    0.315000
 left middle                Random Forest       curl_curl    0.730000
 left   ring                Random Forest       curl_curl    0.600000
 left  pinky                Random Forest       curl_curl    0.610000
right  thumb                Random Forest   straight_curl    0.510000
right  index                Random Forest   straight_curl    0.555000
right middle                Random Forest       curl_curl    0.550000
right   ring Stacking (SVM+RF+KNN+LogReg)   straight_curl    0.742872
right  pinky                Random Forest       curl_curl    0.490000

Per-class probabilities:
 hand finger  p(cur

In [43]:
# ── 7.4 (Optional) Compare every trained classifier on the same CSV ────────────
# Useful for sanity-checking which model agrees with which on a given gesture.

compare_rows = []
for clf_name in active_classifiers:
    df_pred = predict_finger_curls(
        TEST_CSV_PATH,
        classifier_name=clf_name,
        return_proba=False,
    )
    for _, r in df_pred.iterrows():
        compare_rows.append({
            'hand': r['hand'],
            'finger': r['finger'],
            'classifier': clf_name,
            'predicted_class': r['predicted_class'],
        })

compare_df = pd.DataFrame(compare_rows)

# Pivot to: rows = (hand, finger), cols = classifier, values = predicted class
pivot = compare_df.pivot_table(
    index=['hand', 'finger'],
    columns='classifier',
    values='predicted_class',
    aggfunc='first',
)

print(f"\nClassifier comparison for: {os.path.basename(TEST_CSV_PATH)}")
print("=" * 80)
print(pivot.to_string())


Loaded glove_data_L_Static_Double_Closed_Fist_3s_50_2026-05-02_00-23-41.csv → raw shape (92, 164)
Loaded glove_data_L_Static_Double_Closed_Fist_3s_50_2026-05-02_00-23-41.csv → raw shape (92, 164)
Loaded glove_data_L_Static_Double_Closed_Fist_3s_50_2026-05-02_00-23-41.csv → raw shape (92, 164)
Loaded glove_data_L_Static_Double_Closed_Fist_3s_50_2026-05-02_00-23-41.csv → raw shape (92, 164)
Loaded glove_data_L_Static_Double_Closed_Fist_3s_50_2026-05-02_00-23-41.csv → raw shape (92, 164)
Loaded glove_data_L_Static_Double_Closed_Fist_3s_50_2026-05-02_00-23-41.csv → raw shape (92, 164)
Loaded glove_data_L_Static_Double_Closed_Fist_3s_50_2026-05-02_00-23-41.csv → raw shape (92, 164)
Loaded glove_data_L_Static_Double_Closed_Fist_3s_50_2026-05-02_00-23-41.csv → raw shape (92, 164)
Loaded glove_data_L_Static_Double_Closed_Fist_3s_50_2026-05-02_00-23-41.csv → raw shape (92, 164)

Classifier comparison for: glove_data_L_Static_Double_Closed_Fist_3s_50_2026-05-02_00-23-41.csv
classifier           

## 8. PDF Run Report

Generates a styled PDF summarising every parameter and result for this run. Output is saved into `./reports/` with a timestamped filename. Run after Section 6 (the training cells that populate `accuracies` and `per_class_tables_wide`).

In [44]:
# =============================================================================
# 8. GENERATE PDF RUN REPORT
# =============================================================================
# Produces a styled PDF summarising every parameter and result for this run.
# Style mirrors the Static Gesture Classification report.
#
# Requires: reportlab  (auto-installed below if missing)

import os, sys, subprocess, datetime
try:
    import reportlab  # noqa: F401
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', 'reportlab'])

from reportlab.lib import colors
from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import mm
from reportlab.platypus import (
    SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle, PageBreak, KeepTogether
)
from reportlab.lib.enums import TA_LEFT

# ── Output path ───────────────────────────────────────────────────────────────
os.makedirs(REPORT_OUTPUT_DIR, exist_ok=True)
_ts = datetime.datetime.now().strftime('%Y-%m-%d_%H-%M-%S')
_report_path = os.path.join(REPORT_OUTPUT_DIR, f'finger_pose_report_{_ts}.pdf')

# ── Style palette (matches the reference report) ──────────────────────────────
HEADER_BLUE   = colors.HexColor('#1F4E79')
ROW_ALT       = colors.HexColor('#F4F6F9')
TABLE_HEADER  = colors.HexColor('#DDE5EE')
SUMMARY_FILL  = colors.HexColor('#FFF2CC')   # light yellow for macro/mean rows
BORDER_GREY   = colors.HexColor('#BFBFBF')
TEXT_DARK     = colors.HexColor('#222222')

_styles = getSampleStyleSheet()
H1 = ParagraphStyle('H1', parent=_styles['Heading1'],
                    fontName='Helvetica-Bold', fontSize=16, leading=20,
                    textColor=TEXT_DARK, spaceAfter=6)
H2 = ParagraphStyle('H2', parent=_styles['Heading2'],
                    fontName='Helvetica-Bold', fontSize=12, leading=16,
                    textColor=HEADER_BLUE, spaceBefore=10, spaceAfter=4)
SUB = ParagraphStyle('SUB', parent=_styles['Normal'],
                     fontName='Helvetica', fontSize=9, textColor=colors.grey,
                     spaceAfter=8)
BODY = ParagraphStyle('BODY', parent=_styles['Normal'],
                      fontName='Helvetica', fontSize=9, leading=12,
                      textColor=TEXT_DARK)
CELL = ParagraphStyle('CELL', parent=BODY, fontSize=8.5, leading=11)
CELL_B = ParagraphStyle('CELL_B', parent=CELL, fontName='Helvetica-Bold')

def _kv_table(rows, col_widths=(55*mm, 110*mm)):
    """Two-column key/value table — bold left labels, plain right values."""
    data = [[Paragraph(str(k), CELL_B), Paragraph(str(v), CELL)] for k, v in rows]
    t = Table(data, colWidths=col_widths, hAlign='LEFT')
    style = [
        ('VALIGN',       (0, 0), (-1, -1), 'TOP'),
        ('LEFTPADDING',  (0, 0), (-1, -1), 4),
        ('RIGHTPADDING', (0, 0), (-1, -1), 4),
        ('TOPPADDING',   (0, 0), (-1, -1), 3),
        ('BOTTOMPADDING',(0, 0), (-1, -1), 3),
        ('LINEBELOW',    (0, 0), (-1, -2), 0.25, BORDER_GREY),
    ]
    for i in range(len(data)):
        if i % 2 == 1:
            style.append(('BACKGROUND', (0, i), (-1, i), ROW_ALT))
    t.setStyle(TableStyle(style))
    return t

def _data_table(header, rows, col_widths=None, highlight_last=False):
    """Tabular table with header row + zebra stripes; optional yellow last row."""
    data = [[Paragraph(str(c), CELL_B) for c in header]]
    for r in rows:
        data.append([Paragraph(str(c), CELL) for c in r])
    t = Table(data, colWidths=col_widths, hAlign='LEFT', repeatRows=1)
    style = [
        ('BACKGROUND',   (0, 0), (-1, 0), TABLE_HEADER),
        ('VALIGN',       (0, 0), (-1, -1), 'MIDDLE'),
        ('LEFTPADDING',  (0, 0), (-1, -1), 4),
        ('RIGHTPADDING', (0, 0), (-1, -1), 4),
        ('TOPPADDING',   (0, 0), (-1, -1), 3),
        ('BOTTOMPADDING',(0, 0), (-1, -1), 3),
        ('LINEBELOW',    (0, 0), (-1, -1), 0.25, BORDER_GREY),
        ('LINEABOVE',    (0, 0), (-1, 0),  0.5,  BORDER_GREY),
    ]
    for i in range(1, len(data)):
        if i % 2 == 0:
            style.append(('BACKGROUND', (0, i), (-1, i), ROW_ALT))
    if highlight_last and len(data) > 1:
        last = len(data) - 1
        style.append(('BACKGROUND', (0, last), (-1, last), SUMMARY_FILL))
        # bold the entire last row
        for ci in range(len(header)):
            data[last][ci] = Paragraph(f"<b>{data[last][ci].text}</b>", CELL_B)
    t.setStyle(TableStyle(style))
    return t

def _h2(text):
    return Paragraph(text, H2)

# ── Pull all parameters straight from globals() ───────────────────────────────
def _g(name, default='—'):
    return globals().get(name, default)

# Resolve actual data path & counts
_data_root_abs = os.path.abspath(os.path.expanduser(_g('DATA_ROOT', '')))
_class_names = sorted(set(globals().get('labels_raw', [])))
_trials_total = len(globals().get('labels_raw', []))

# Per-class trial counts
from collections import Counter as _Counter
_class_counts = _Counter(globals().get('labels_raw', []))

# Sensor column count
_sensor_cols = globals().get('SENSOR_COLS', [])

# Modality summary string
_modal_parts = []
if _g('USE_YAW_PITCH_ROLL', False): _modal_parts.append('YPR')
if _g('USE_QUATERNIONS', False):    _modal_parts.append('Quaternions')
if _g('USE_ACCELEROMETER', False):  _modal_parts.append('Accel')
if _g('USE_FLEX_SENSORS', False):   _modal_parts.append('Flex')

# Feature dimensionality (use first available finger's X_train)
_feature_dim = None
_finger_data_aug = globals().get('finger_data_aug', {})
if _finger_data_aug:
    _first_key = next(iter(_finger_data_aug))
    _xt = _finger_data_aug[_first_key].get('X_train')
    if _xt is not None and hasattr(_xt, 'shape'):
        _feature_dim = _xt.shape[1]

# Story assembly
story = []
story.append(Paragraph('Finger Pose Classification — Run Report', H1))
story.append(Paragraph(
    f"Generated {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}", SUB))

# ── Data paths ────────────────────────────────────────────────────────────────
story.append(_h2('Data paths'))
_kv_rows = [
    ('Data root', f"{_data_root_abs} · {_trials_total} trials total"),
    ('Class folders', ', '.join(_class_names) if _class_names else '—'),
    ('Report output dir', os.path.abspath(REPORT_OUTPUT_DIR)),
]
for _cls, _cnt in sorted(_class_counts.items()):
    _kv_rows.append((f'Trials · {_cls}', f'{_cnt}'))
story.append(_kv_table(_kv_rows))

# ── Sensor selection ──────────────────────────────────────────────────────────
story.append(_h2('Sensor selection'))
story.append(_kv_table([
    ('Hands',         ', '.join(_g('INCLUDE_HANDS', []))),
    ('Segments',      ', '.join(_g('INCLUDE_SEGMENTS', []))),
    ('Modalities',    ', '.join(_modal_parts) if _modal_parts else 'none'),
    ('Sensor columns', f'{len(_sensor_cols)} channels'),
    ('Fingers',       ', '.join(_g('FINGERS', []))),
    ('Classes',       f"{len(_class_names)}: {', '.join(_class_names)}"
                      if _class_names else '—'),
]))

# ── Preprocessing, features & split ───────────────────────────────────────────
story.append(_h2('Preprocessing, features &amp; split'))
_bw_on  = bool(_g('APPLY_BUTTERWORTH', False))
_bw_str = (f"on · cutoff {_g('BUTTERWORTH_CUTOFF')} Hz · "
           f"order {_g('BUTTERWORTH_ORDER')} · fs {_g('SAMPLING_RATE_HZ')} Hz"
           if _bw_on else 'off')
_feat_str = f"{_g('FEATURE_MODE')}"
if _feature_dim is not None:
    _feat_str += f" → feature dim {_feature_dim}"
story.append(_kv_table([
    ('Resample steps', f"{_g('RESAMPLE_TO_N_STEPS')}"),
    ('Butterworth',    _bw_str),
    ('Normalisation',  f"{_g('NORMALISATION')}"),
    ('Feature mode',   _feat_str),
    ('Split / seed / CV',
        f"test_size={_g('TEST_SIZE')} · random_state={_g('RANDOM_STATE')} · "
        f"CV folds={_g('CV_FOLDS')}"),
    ('Trials per finger dataset',
        f"{len(_finger_data_aug[next(iter(_finger_data_aug))]['raw_train']) + len(_finger_data_aug[next(iter(_finger_data_aug))]['raw_test'])}"
        f" (train+test, post-aug train)" if _finger_data_aug else '—'),
]))

# ── Augmentation ──────────────────────────────────────────────────────────────
# These are the literals used in the augment_raw_trials() call in cell 19.
# Update here if you change them in the call.
story.append(_h2('Augmentation (training only)'))
story.append(_kv_table([
    ('Copies per train trial', '3'),
    ('Gaussian noise',         'on · sigma=0.02 (per-channel std-scaled)'),
    ('Time warp',              'on · max_scale=0.1'),
    ('Amplitude scaling',      'off · range=(0.9, 1.1)'),
]))

# ── Classifiers list ──────────────────────────────────────────────────────────
_active = globals().get('active_classifiers', {})
story.append(_h2('Active classifiers'))
_clf_rows = []
for _n, _c in _active.items():
    _clf_rows.append([_n, type(_c).__name__])
story.append(_data_table(['Algorithm', 'Class'], _clf_rows,
                         col_widths=(70*mm, 95*mm)))

# Hyperparameters per classifier (compact)
story.append(_h2('Classifier hyperparameters'))
for _n, _c in _active.items():
    _params = _c.get_params(deep=False)
    # Drop obviously verbose nested estimators for readability
    _clean = {k: v for k, v in _params.items()
              if not k.endswith('estimators') and not k.startswith('estimator__')}
    # Render as compact key=value chunks
    _chunks = [f"{k}={v}" for k, v in _clean.items()]
    _txt = ' · '.join(_chunks) if _chunks else '—'
    story.append(_kv_table([(_n, _txt)], col_widths=(55*mm, 110*mm)))
    story.append(Spacer(1, 2))

# ── Results: overall accuracy per finger dataset × classifier ─────────────────
_accuracies = globals().get('accuracies', {})
if _accuracies:
    story.append(PageBreak())
    story.append(_h2('Per-dataset test accuracy'))
    story.append(Paragraph(
        "Rows = (hand, finger) dataset. Columns = classifier. "
        "Last row = mean across datasets.", BODY))
    story.append(Spacer(1, 4))

    _clf_names = list(next(iter(_accuracies.values())).keys())
    _header = ['Dataset'] + _clf_names
    _rows = []
    for _k, _row in _accuracies.items():
        _label = f"{_k[0]} · {_k[1]}" if isinstance(_k, tuple) else str(_k)
        _rows.append([_label] + [f"{_row.get(n, float('nan')):.3f}"
                                 for n in _clf_names])
    # mean row
    import numpy as _np
    _means = []
    for n in _clf_names:
        _vals = [_row.get(n) for _row in _accuracies.values() if _row.get(n) is not None]
        _means.append(f"{_np.mean(_vals):.3f}" if _vals else '—')
    _rows.append(['Mean'] + _means)

    _ncols = len(_header)
    _wfirst = 35*mm
    _wrest = (165*mm - _wfirst) / (_ncols - 1)
    story.append(_data_table(_header, _rows,
                             col_widths=[_wfirst] + [_wrest]*(_ncols-1),
                             highlight_last=True))

# ── Per-class accuracy tables, one per finger dataset ─────────────────────────
_per_class_wide = globals().get('per_class_tables_wide', {})
if _per_class_wide:
    story.append(_h2('Per-class accuracy by dataset'))
    story.append(Paragraph(
        "Each table: rows = class, columns = classifier. "
        "Values are per-class accuracy on the test set.", BODY))
    story.append(Spacer(1, 4))

    for _k, _df in _per_class_wide.items():
        _label = f"{_k[0]} · {_k[1]}" if isinstance(_k, tuple) else str(_k)
        story.append(Paragraph(f"<b>{_label}</b>", BODY))
        story.append(Spacer(1, 2))

        _cols = list(_df.columns)
        _header = _cols
        _rows = []
        for _, _r in _df.iterrows():
            _rows.append([_r[_cols[0]]] +
                         [f"{_r[c]:.3f}" if isinstance(_r[c], (int, float)) else str(_r[c])
                          for c in _cols[1:]])

        _ncols = len(_header)
        _wfirst = 30*mm
        _wrest = (165*mm - _wfirst) / max(1, (_ncols - 1))
        story.append(_data_table(_header, _rows,
                                 col_widths=[_wfirst] + [_wrest]*(_ncols-1)))
        story.append(Spacer(1, 6))

# ── Build the PDF ─────────────────────────────────────────────────────────────
_doc = SimpleDocTemplate(
    _report_path, pagesize=A4,
    leftMargin=15*mm, rightMargin=15*mm,
    topMargin=15*mm, bottomMargin=15*mm,
    title='Finger Pose Classification — Run Report',
)
_doc.build(story)
print(f'Report saved: {_report_path}')


Report saved: /home/jestin/ThesisRepo/ML/NewReports/FingerPoseReports/FourPose/finger_pose_report_2026-05-21_01-04-12.pdf
